# BERTScore で意味的な近さを評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_evaluation/bertscore.ipynb)

NLP クックブック 13-04 ／ 評価指標と統計的検定

表現が違っても意味が同じなら高く評価したい。n-gram の一致ではなく、文脈化埋め込みの類似度で測る。

In [1]:
!pip install -q bert-score

In [2]:
from bert_score import score

refs = ["猫はマットの上に座っています。", "彼は昨日、東京に行きました。", "この本は非常に面白い。", "この本は非常に面白い。"]
hyps = ["猫がマットの上に座っている。", "昨日、彼は東京へ出かけた。", "この書籍はとても興味深い。", "この本は非常に面白くない。"]

P, R, F = score(hyps, refs, lang="ja", verbose=False)   # lang="ja" では bert-base-multilingual-cased が使われる
for h, r, f in zip(hyps, refs, F.tolist()):
    print(f"{f:.3f}  {h}  /  {r}")
print("平均 F1:", F.mean().item())

0.895  猫がマットの上に座っている。  /  猫はマットの上に座っています。
0.890  昨日、彼は東京へ出かけた。  /  彼は昨日、東京に行きました。
0.815  この書籍はとても興味深い。  /  この本は非常に面白い。
0.963  この本は非常に面白くない。  /  この本は非常に面白い。
平均 F1: 0.8908554315567017


In [3]:
# 無関係な文同士でも 0.7 近くになる（値の範囲が狭い）ことに注意。設定を表すハッシュも取得できる
(P, R, F), setting = score(["今日は天気が良い。"], ["会議は午後 3 時に始まります。"], lang="ja", verbose=False, return_hash=True)
print(round(F.item(), 3), setting)

0.688 bert-base-multilingual-cased_L9_no-idf_version=0.3.12(hug_trans=5.22.1)


- 言い換え（「本 / 書籍」「面白い / 興味深い」）にも高いスコアが付く。一方で、否定のような小さいが決定的な違い（4 つ目の例）には鈍感。
- 値はモデルと層に依存する。報告するときはハッシュを添える。英語などでは `rescale_with_baseline=True` で値の範囲を広げて読みやすくできる。
- 翻訳評価では、人手評価で学習した COMET などのほうが人間の判断との相関が高い。